In [0]:
-- ================================================================
-- trusted.facility_trust_scores
-- Grain: one row per facility (same as facilities_enriched)
-- Source: benefits_navigator.trusted.facilities_enriched
--
-- Dimensions:
--   clinical_score      40% — does it do what it claims?
--   completeness_score  25% — did they fill in their profile?
--   digital_score       20% — are they active and findable?
--   geo_score           15% — can a patient actually reach them?
--
-- Output trust_tier:
--   high        >= 70
--   medium      >= 45
--   low         >= 20
--   unverified  <  20  (too little data to assess)
--
-- Never surface raw numbers to patients.
-- Claude narrates trust_tier + top_trust_signals in plain language.
-- ================================================================

CREATE OR REPLACE TABLE `benefits_navigator`.`trusted`.`facility_trust_scores`
USING DELTA
COMMENT 'Per-facility trust scores derived from facilities_enriched. Four sub-scores compose into a weighted trust_score (0-100) and a tier. Built for Track 1 Facility Trust Desk.'
AS
WITH

-- ── 1. Clinical credibility (0–100) ─────────────────────────
-- Signals: specialties, capability, equipment, procedures, number_doctors
-- Question: does the facility have documented clinical substance?
clinical AS (
  SELECT
    unique_id,

    -- Each signal adds points; cap at 100
    LEAST(100, (

      -- Specialties present and non-trivial (>10 chars = real content)
      CASE
        WHEN specialties IS NULL                        THEN 0
        WHEN LENGTH(TRIM(specialties)) < 10            THEN 10
        ELSE                                                 30
      END

      -- Capability text present and substantive
      + CASE
        WHEN capability IS NULL                         THEN 0
        WHEN LENGTH(TRIM(capability)) < 20             THEN 8
        ELSE                                                 25
      END

      -- Equipment listed (signals physical infrastructure)
      + CASE
        WHEN equipment IS NULL                          THEN 0
        ELSE                                                 20
      END

      -- Procedures documented
      + CASE
        WHEN procedures IS NULL                         THEN 0
        WHEN LENGTH(TRIM(procedures)) < 10             THEN 5
        ELSE                                                 15
      END

      -- Verified doctor count
      + CASE
        WHEN number_doctors IS NULL                     THEN 0
        WHEN number_doctors = 0                        THEN 0
        WHEN number_doctors BETWEEN 1 AND 5            THEN 5
        WHEN number_doctors BETWEEN 6 AND 20           THEN 8
        ELSE                                                 10
      END

    )) AS clinical_score,

    -- Human-readable signals for Claude to narrate
    ARRAY_JOIN(FILTER(ARRAY(
      CASE WHEN specialties IS NOT NULL
           THEN 'documented specialties' END,
      CASE WHEN capability IS NOT NULL AND LENGTH(TRIM(capability)) >= 20
           THEN 'detailed capability profile' END,
      CASE WHEN equipment IS NOT NULL
           THEN 'equipment inventory listed' END,
      CASE WHEN procedures IS NOT NULL
           THEN 'procedures documented' END,
      CASE WHEN number_doctors > 0
           THEN CONCAT(CAST(number_doctors AS STRING), ' doctors on record') END
    ), x -> x IS NOT NULL), ', ') AS clinical_signals,

    ARRAY_JOIN(FILTER(ARRAY(
      CASE WHEN specialties IS NULL     THEN 'no specialties listed'    END,
      CASE WHEN capability IS NULL      THEN 'no capability detail'     END,
      CASE WHEN equipment IS NULL       THEN 'no equipment listed'      END,
      CASE WHEN number_doctors IS NULL
            OR number_doctors = 0       THEN 'doctor count unverified'  END
    ), x -> x IS NOT NULL), ', ') AS clinical_concerns

  FROM `benefits_navigator`.`trusted`.`facilities_enriched`
),

-- ── 2. Profile completeness (0–100) ─────────────────────────
-- Signals: contact info, address, description, website
-- Question: did someone actually maintain this profile?
completeness AS (
  SELECT
    unique_id,

    LEAST(100, (

      -- Phone: official is better than scraped
      CASE
        WHEN official_phone  IS NOT NULL                THEN 25
        WHEN phone_numbers   IS NOT NULL                THEN 15
        ELSE                                                  0
      END

      -- Address completeness
      + CASE
        WHEN address_line1 IS NOT NULL
         AND address_city  IS NOT NULL                  THEN 25
        WHEN address_city  IS NOT NULL                  THEN 12
        ELSE                                                  0
      END

      -- Website
      + CASE
        WHEN official_website IS NOT NULL               THEN 25
        WHEN websites         IS NOT NULL               THEN 12
        ELSE                                                  0
      END

      -- Description present and meaningful
      + CASE
        WHEN description IS NULL                        THEN 0
        WHEN LENGTH(TRIM(description)) < 30            THEN 10
        ELSE                                                 25
      END

    )) AS completeness_score,

    ARRAY_JOIN(FILTER(ARRAY(
      CASE WHEN official_phone IS NOT NULL
           THEN 'official phone verified'              END,
      CASE WHEN official_website IS NOT NULL
           THEN 'official website listed'              END,
      CASE WHEN address_line1 IS NOT NULL
            AND address_city  IS NOT NULL
           THEN 'full address on record'               END,
      CASE WHEN description IS NOT NULL
            AND LENGTH(TRIM(description)) >= 30
           THEN 'facility description provided'        END
    ), x -> x IS NOT NULL), ', ') AS completeness_signals,

    ARRAY_JOIN(FILTER(ARRAY(
      CASE WHEN official_phone IS NULL
            AND phone_numbers  IS NULL
           THEN 'no contact number'                    END,
      CASE WHEN official_website IS NULL
            AND websites IS NULL
           THEN 'no website'                           END,
      CASE WHEN address_line1 IS NULL
           THEN 'incomplete address'                   END,
      CASE WHEN description IS NULL
           THEN 'no description'                       END
    ), x -> x IS NOT NULL), ', ') AS completeness_concerns

  FROM `benefits_navigator`.`trusted`.`facilities_enriched`
),

-- ── 3. Digital presence (0–100) ──────────────────────────────
-- Signals: social media count, recency, followers, staff profiles
-- Question: is this facility actively maintained and findable?
digital AS (
  SELECT
    unique_id,

    LEAST(100, (

      -- Social media platform spread (each platform = ~15pts, cap at 3)
      LEAST(45, COALESCE(social_media_count, 0) * 15)

      -- Recency of page update
      + CASE
        WHEN recency_of_page_update IS NULL             THEN 0
        -- Assume recency stored as days since update (adjust if format differs)
        WHEN TRY_CAST(recency_of_page_update AS INT) <= 90   THEN 25
        WHEN TRY_CAST(recency_of_page_update AS INT) <= 365  THEN 15
        ELSE                                                   5
      END

      -- Follower count: log-scaled so large accounts don't dominate
      + CASE
        WHEN COALESCE(follower_count, 0) = 0           THEN 0
        WHEN follower_count < 100                      THEN 5
        WHEN follower_count < 1000                     THEN 10
        WHEN follower_count < 10000                    THEN 15
        ELSE                                                20
      END

      -- Verified affiliated staff profiles
      + CASE
        WHEN has_affiliated_staff = TRUE               THEN 10
        ELSE                                                 0
      END

    )) AS digital_score,

    ARRAY_JOIN(FILTER(ARRAY(
      CASE WHEN COALESCE(social_media_count, 0) >= 2
           THEN CONCAT(CAST(social_media_count AS STRING), ' social platforms') END,
      CASE WHEN has_affiliated_staff = TRUE
           THEN 'staff profiles linked'                END,
      CASE WHEN TRY_CAST(recency_of_page_update AS INT) <= 90
           THEN 'profile updated recently'             END,
      CASE WHEN COALESCE(follower_count, 0) >= 1000
           THEN CONCAT(CAST(follower_count AS STRING), ' followers') END
    ), x -> x IS NOT NULL), ', ') AS digital_signals,

    ARRAY_JOIN(FILTER(ARRAY(
      CASE WHEN COALESCE(social_media_count, 0) = 0
           THEN 'no social media presence'             END,
      CASE WHEN has_affiliated_staff IS NULL
            OR has_affiliated_staff = FALSE
           THEN 'no staff profiles'                    END,
      CASE WHEN recency_of_page_update IS NULL
           THEN 'page update date unknown'             END
    ), x -> x IS NOT NULL), ', ') AS digital_concerns

  FROM `benefits_navigator`.`trusted`.`facilities_enriched`
),

-- ── 4. Geo verifiability (0–100) ─────────────────────────────
-- Signals: geo_match_method, coords_missing, multi_district_flag, zip_quality
-- Question: can a patient physically find and navigate to this facility?
geo AS (
  SELECT
    unique_id,

    LEAST(100, (

      -- Primary: how well did the pincode resolve?
      CASE geo_match_method
        WHEN 'pincode_exact'  THEN 50
        WHEN 'pincode_alias'  THEN 40
        WHEN 'zip_not_in_lookup' THEN 15
        WHEN 'no_zip'         THEN 0
        ELSE                       0
      END

      -- Coordinates available (patient can use maps)
      + CASE
        WHEN coords_missing = FALSE                     THEN 30
        ELSE                                                  0
      END

      -- Clean zip on file
      + CASE
        WHEN zip_quality = 'ok'                         THEN 15
        ELSE                                                  0
      END

      -- Penalty: ambiguous multi-district pincode
      + CASE
        WHEN multi_district_flag = TRUE                 THEN -10
        ELSE                                                   5
      END

    )) AS geo_score,

    ARRAY_JOIN(FILTER(ARRAY(
      CASE WHEN geo_match_method IN ('pincode_exact','pincode_alias')
           THEN 'pincode verified'                     END,
      CASE WHEN coords_missing = FALSE
           THEN 'GPS coordinates available'            END,
      CASE WHEN zip_quality = 'ok'
           THEN 'valid postcode on file'               END
    ), x -> x IS NOT NULL), ', ') AS geo_signals,

    ARRAY_JOIN(FILTER(ARRAY(
      CASE WHEN geo_match_method IN ('no_zip','zip_not_in_lookup')
           THEN 'location could not be verified'       END,
      CASE WHEN coords_missing = TRUE
           THEN 'no GPS coordinates'                   END,
      CASE WHEN multi_district_flag = TRUE
           THEN 'postcode spans multiple districts'    END
    ), x -> x IS NOT NULL), ', ') AS geo_concerns

  FROM `benefits_navigator`.`trusted`.`facilities_enriched`
),

-- ── 5. Compose weighted score ────────────────────────────────
composed AS (
  SELECT
    fe.unique_id,
    fe.name,
    fe.facility_type,
    fe.operator_type,
    fe.district_norm,
    fe.state_norm,
    fe.latitude,
    fe.longitude,
    fe.is_dark_facility,

    -- Sub-scores (each 0–100)
    ROUND(c.clinical_score,    1) AS clinical_score,
    ROUND(p.completeness_score,1) AS completeness_score,
    ROUND(d.digital_score,     1) AS digital_score,
    ROUND(g.geo_score,         1) AS geo_score,

    -- Weighted composite (0–100)
    ROUND(
      (c.clinical_score     * 0.40) +
      (p.completeness_score * 0.25) +
      (d.digital_score      * 0.20) +
      (g.geo_score          * 0.15)
    , 1) AS trust_score,

    -- Score confidence: how many sub-dimensions had real data?
    -- Low confidence = score is based on sparse signals
    (
      CASE WHEN c.clinical_score    > 0 THEN 1 ELSE 0 END +
      CASE WHEN p.completeness_score > 0 THEN 1 ELSE 0 END +
      CASE WHEN d.digital_score     > 0 THEN 1 ELSE 0 END +
      CASE WHEN g.geo_score         > 0 THEN 1 ELSE 0 END
    ) AS dimensions_with_data,

    -- Signal and concern strings for Claude
    c.clinical_signals,
    p.completeness_signals,
    d.digital_signals,
    g.geo_signals,
    c.clinical_concerns,
    p.completeness_concerns,
    d.digital_concerns,
    g.geo_concerns

  FROM `benefits_navigator`.`trusted`.`facilities_enriched` fe
  LEFT JOIN clinical    c ON fe.unique_id = c.unique_id
  LEFT JOIN completeness p ON fe.unique_id = p.unique_id
  LEFT JOIN digital     d ON fe.unique_id = d.unique_id
  LEFT JOIN geo         g ON fe.unique_id = g.unique_id
)

-- ── 6. Final SELECT with derived fields ──────────────────────
SELECT
  unique_id,
  name,
  facility_type,
  operator_type,
  district_norm,
  state_norm,
  latitude,
  longitude,
  is_dark_facility,

  -- Sub-scores
  clinical_score,
  completeness_score,
  digital_score,
  geo_score,

  -- Composite
  trust_score,

  -- Tier — what Claude actually uses
  CASE
    WHEN is_dark_facility = TRUE              THEN 'unverified'
    WHEN trust_score >= 70                    THEN 'high'
    WHEN trust_score >= 45                    THEN 'medium'
    WHEN trust_score >= 20                    THEN 'low'
    ELSE                                           'unverified'
  END AS trust_tier,

  -- Confidence flag
  CASE
    WHEN dimensions_with_data >= 3            THEN 'high'
    WHEN dimensions_with_data = 2             THEN 'medium'
    ELSE                                           'low'
  END AS score_confidence,

  dimensions_with_data,

  -- Top positive signals (comma-separated, for Claude prompt injection)
  CONCAT_WS(', ',
    NULLIF(clinical_signals,     ''),
    NULLIF(completeness_signals, ''),
    NULLIF(digital_signals,      ''),
    NULLIF(geo_signals,          '')
  ) AS top_trust_signals,

  -- Top concerns (comma-separated, for Claude prompt injection)
  CONCAT_WS(', ',
    NULLIF(clinical_concerns,     ''),
    NULLIF(completeness_concerns, ''),
    NULLIF(digital_concerns,      ''),
    NULLIF(geo_concerns,          '')
  ) AS top_concerns,

  -- Timestamp so you know when scores were last computed
  CURRENT_TIMESTAMP() AS scored_at

FROM composed;

In [0]:
-- ================================================================
-- Validation queries (uncomment and run after CREATE)
-- ================================================================

-- Trust tier distribution
-- SELECT trust_tier, score_confidence,
--   COUNT(*) AS n,
--   ROUND(AVG(trust_score), 1) AS avg_score,
--   ROUND(MIN(trust_score), 1) AS min_score,
--   ROUND(MAX(trust_score), 1) AS max_score
-- FROM `benefits_navigator`.`trusted`.`facility_trust_scores`
-- GROUP BY trust_tier, score_confidence
-- ORDER BY avg_score DESC;

-- Sample high-trust facilities
SELECT name, district_norm, state_norm, trust_score, trust_tier,
  top_trust_signals
FROM `benefits_navigator`.`trusted`.`facility_trust_scores`
WHERE trust_tier = 'high'
ORDER BY trust_score DESC
LIMIT 20;

-- Sample of what Claude will see per facility
-- SELECT name, trust_tier, score_confidence,
--   clinical_score, completeness_score, digital_score, geo_score,
--   top_trust_signals, top_concerns
-- FROM `benefits_navigator`.`trusted`.`facility_trust_scores`
-- WHERE trust_tier IS NOT NULL
-- ORDER BY trust_score DESC
-- LIMIT 10;

-- Sub-score null check
-- SELECT
--   SUM(CASE WHEN clinical_score    IS NULL THEN 1 ELSE 0 END) AS clinical_nulls,
--   SUM(CASE WHEN completeness_score IS NULL THEN 1 ELSE 0 END) AS completeness_nulls,
--   SUM(CASE WHEN digital_score     IS NULL THEN 1 ELSE 0 END) AS digital_nulls,
--   SUM(CASE WHEN geo_score         IS NULL THEN 1 ELSE 0 END) AS geo_nulls,
--   COUNT(*) AS total
-- FROM `benefits_navigator`.`trusted`.`facility_trust_scores`;